# Capítulo 9 — Trocadores de Calor

**Autor:** Jader Lugon Junior  
**Livro:** Fenômenos de Transporte: Fundamentos e Modelagem Computacional  
**Repositório:** [github.com/JaderLugon/fenomenos-transporte-livro](https://github.com/JaderLugon/fenomenos-transporte-livro)

---

## 🎯 Objetivos de Aprendizagem

Ao final deste notebook, você será capaz de:

1. **Calcular** a carga térmica em trocadores de calor pelo balanço de energia
2. **Aplicar** o método LMTD (Log Mean Temperature Difference) para dimensionamento
3. **Aplicar** o método ε-NTU para análise de trocadores existentes
4. **Calcular** o coeficiente global de transferência de calor $U$
5. **Interpretar** correlações convectivas (Dittus-Boelter, Zukauskas)
6. **Analisar** o efeito da incrustação no desempenho de trocadores

---

## 📚 Conteúdo Teórico Resumido

### 9.1 Balanço de Energia

Para um trocador em regime permanente (sem perdas para o ambiente):

$$
Q = \dot{m}_h \, c_{p,h} \, (T_{h,in} - T_{h,out}) = \dot{m}_c \, c_{p,c} \, (T_{c,out} - T_{c,in})
$$

### 9.2 Método LMTD

$$
Q = U \cdot A \cdot F \cdot \Delta T_{lm}
$$

onde:
$$
\Delta T_{lm} = \frac{\Delta T_1 - \Delta T_2}{\ln(\Delta T_1 / \Delta T_2)}
$$

- $\Delta T_1, \Delta T_2$: diferenças de temperatura nas extremidades
- $F$: fator de correção (F = 1 para contracorrente puro)

### 9.3 Método ε-NTU

$$
\varepsilon = \frac{Q}{Q_{\max}}, \quad Q_{\max} = C_{\min} \cdot (T_{h,in} - T_{c,in})
$$

$$
\mathrm{NTU} = \frac{U \cdot A}{C_{\min}}, \quad C_r = \frac{C_{\min}}{C_{\max}}
$$

Para contracorrente:
$$
\varepsilon = \frac{1 - \exp[-\mathrm{NTU}(1 - C_r)]}{1 - C_r \exp[-\mathrm{NTU}(1 - C_r)]}
$$

### 9.4 Coeficiente Global $U$

$$
\frac{1}{U \cdot A} = \frac{1}{h_i \cdot A_i} + R_{f,i} + \frac{\ln(D_e/D_i)}{2\pi k L} + R_{f,e} + \frac{1}{h_e \cdot A_e}
$$

### 9.5 Correlações Convectivas

**Dittus-Boelter** (escoamento turbulento em tubos):
$$
\mathrm{Nu} = 0,023 \, \mathrm{Re}^{0,8} \, \mathrm{Pr}^n, \quad n = \begin{cases} 0,4 & \text{aquecimento} \\ 0,3 & \text{resfriamento} \end{cases}
$$

Válida para $\mathrm{Re} > 10^4$ e $0,6 < \mathrm{Pr} < 160$.

---

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import sys
import os

# Adiciona o diretório raiz ao path para importar módulos
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

# Configuração de plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✅ Ambiente configurado com sucesso!")
print(f"📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}")

---

## 🔥 Seção 1: Balanço de Energia e Carga Térmica

### Exercício 1.1: Carga Térmica e Temperatura de Saída

Um trocador resfria óleo hidráulico ($\dot{m}_h = 15$ kg/s, $c_{p,h} = 2000$ J/(kg·K), $T_{h,in} = 60$°C, $T_{h,out} = 45$°C) usando água ($\dot{m}_c = 12$ kg/s, $c_{p,c} = 4180$ J/(kg·K), $T_{c,in} = 20$°C). Calcule a carga térmica e a temperatura de saída da água.

In [ ]:
# ============================================================
# EXERCÍCIO 1.1: Balanço de Energia
# ============================================================

print("=" * 60)
print("EXERCÍCIO 1.1: BALANÇO DE ENERGIA EM TROCADOR")
print("=" * 60)

# Dados do óleo (fluido quente)
m_h = 15.0         # kg/s
cp_h = 2000.0      # J/(kg·K)
T_h_in = 60.0      # °C
T_h_out = 45.0     # °C

# Dados da água (fluido frio)
m_c = 12.0         # kg/s
cp_c = 4180.0      # J/(kg·K)
T_c_in = 20.0      # °C

# Cálculo da carga térmica (lado quente)
Q = m_h * cp_h * (T_h_in - T_h_out)

# Temperatura de saída da água (balanço de energia)
T_c_out = T_c_in + Q / (m_c * cp_c)

# Capacidades térmicas
C_h = m_h * cp_h
C_c = m_c * cp_c

print(f"\n📊 Dados do problema:")
print(f"  • Óleo: ṁ = {m_h} kg/s, c_p = {cp_h} J/(kg·K)")
print(f"    T_in = {T_h_in}°C, T_out = {T_h_out}°C")
print(f"  • Água: ṁ = {m_c} kg/s, c_p = {cp_c} J/(kg·K)")
print(f"    T_in = {T_c_in}°C")

print(f"\n🧮 Cálculo:")
print(f"  • Q = ṁ_h · c_p,h · (T_h,in - T_h,out)")
print(f"  • Q = {m_h} × {cp_h} × ({T_h_in} - {T_h_out})")
print(f"  • Q = {Q:.0f} W = {Q/1000:.1f} kW")
print(f"\n  • T_c,out = T_c,in + Q/(ṁ_c · c_p,c)")
print(f"  • T_c,out = {T_c_in} + {Q:.0f}/({m_c} × {cp_c})")
print(f"  • T_c,out = {T_c_out:.2f}°C")

print(f"\n🔍 Capacidades térmicas:")
print(f"  • C_h = {C_h:.0f} W/K")
print(f"  • C_c = {C_c:.0f} W/K")
print(f"  • Razão: C_c/C_h = {C_c/C_h:.2f}")

print(f"\n💡 Interpretação:")
print(f"  A água tem maior capacidade térmica ({C_c:.0f} W/K vs {C_h:.0f} W/K),")
print(f"  por isso sofre menor variação de temperatura ({T_c_out - T_c_in:.1f}°C")
print(f"  vs {T_h_in - T_h_out:.1f}°C do óleo).")

---

## 🌡️ Seção 2: Método LMTD

### Exercício 2.1: LMTD e Área Necessária

Para o trocador do exercício anterior, calcule a LMTD e a área necessária, considerando $U = 600$ W/(m²·K) e trocador em contracorrente.

In [ ]:
# ============================================================
# EXERCÍCIO 2.1: LMTD e Área
# ============================================================

print("=" * 60)
print("EXERCÍCIO 2.1: LMTD E ÁREA NECESSÁRIA")
print("=" * 60)

# Coeficiente global
U = 600.0  # W/(m²·K)

# Diferenças de temperatura nas extremidades (contracorrente)
dT1 = T_h_in - T_c_out    # extremidade quente
dT2 = T_h_out - T_c_in    # extremidade fria

# LMTD
if abs(dT1 - dT2) < 1e-6:
    LMTD = (dT1 + dT2) / 2
else:
    LMTD = (dT1 - dT2) / np.log(dT1 / dT2)

# Área necessária (F = 1 para contracorrente)
F = 1.0
A = Q / (U * F * LMTD)

print(f"\n📊 Diferenças de temperatura:")
print(f"  • ΔT₁ = T_h,in - T_c,out = {T_h_in} - {T_c_out:.2f} = {dT1:.2f}°C")
print(f"  • ΔT₂ = T_h,out - T_c,in = {T_h_out} - {T_c_in} = {dT2:.2f}°C")

print(f"\n🧮 LMTD:")
print(f"  • LMTD = (ΔT₁ - ΔT₂) / ln(ΔT₁/ΔT₂)")
print(f"  • LMTD = ({dT1:.2f} - {dT2:.2f}) / ln({dT1:.2f}/{dT2:.2f})")
print(f"  • LMTD = {LMTD:.2f}°C")

print(f"\n📐 Área necessária:")
print(f"  • A = Q / (U · F · LMTD)")
print(f"  • A = {Q:.0f} / ({U} × {F} × {LMTD:.2f})")
print(f"  • A = {A:.2f} m²")

# Comparação com média aritmética
dT_arith = (dT1 + dT2) / 2
erro_LMTD = abs(dT_arith - LMTD) / LMTD * 100

print(f"\n🔍 Comparação com média aritmética:")
print(f"  • Média aritmética: {dT_arith:.2f}°C")
print(f"  • LMTD: {LMTD:.2f}°C")
print(f"  • Erro: {erro_LMTD:.2f}%")

print(f"\n💡 Interpretação:")
print(f"  Como ΔT₁/ΔT₂ = {dT1/dT2:.2f} < 1,5, a média aritmética")
print(f"  seria aceitável (erro < 1,5%), mas usamos LMTD por ser exata.")

---

## 📊 Seção 3: Método ε-NTU

### Exercício 3.1: Efetividade e NTU

Para um trocador com $U = 450$ W/(m²·K), $A = 12$ m², $C_{\min} = 2000$ W/K, $C_r = 0,5$, calcule $\varepsilon$ e $Q$ para $T_{h,in} - T_{c,in} = 80$ K.

In [ ]:
# ============================================================
# EXERCÍCIO 3.1: Método ε-NTU
# ============================================================

print("=" * 60)
print("EXERCÍCIO 3.1: MÉTODO ε-NTU")
print("=" * 60)

# Dados
U_ex3 = 450.0        # W/(m²·K)
A_ex3 = 12.0         # m²
C_min = 2000.0       # W/K
C_r = 0.5            # adimensional
delta_T_max = 80.0   # K

# NTU
NTU = U_ex3 * A_ex3 / C_min

# Efetividade (contracorrente)
exp_term = np.exp(-NTU * (1 - C_r))
epsilon = (1 - exp_term) / (1 - C_r * exp_term)

# Carga térmica
Q_max = C_min * delta_T_max
Q_ex3 = epsilon * Q_max

print(f"\n📊 Dados do problema:")
print(f"  • U = {U_ex3} W/(m²·K)")
print(f"  • A = {A_ex3} m²")
print(f"  • C_min = {C_min:.0f} W/K")
print(f"  • C_r = {C_r}")
print(f"  • ΔT_max = {delta_T_max} K")

print(f"\n🧮 Cálculo:")
print(f"  • NTU = U·A/C_min = {U_ex3}×{A_ex3}/{C_min:.0f} = {NTU:.2f}")
print(f"  • ε = (1 - exp[-NTU(1-C_r)]) / (1 - C_r·exp[-NTU(1-C_r)])")
print(f"  • ε = {epsilon:.3f}")
print(f"\n  • Q_max = C_min·ΔT_max = {C_min:.0f}×{delta_T_max} = {Q_max:.0f} W")
print(f"  • Q = ε·Q_max = {epsilon:.3f}×{Q_max:.0f} = {Q_ex3:.0f} W")

print(f"\n💡 Interpretação:")
print(f"  A efetividade ε = {epsilon:.3f} indica que o trocador transfere")
print(f"  {epsilon*100:.1f}% do calor máximo possível.")
print(f"  NTU = {NTU:.2f} indica boa capacidade de transferência.")

---

## 🧱 Seção 4: Coeficiente Global U

### Exercício 4.1: Resistências em Série

Calcule o coeficiente global $U$ para um tubo de aço ($k = 45$ W/(m·K), $D_i = 20$ mm, $D_e = 25$ mm) com:
- $h_i = 3500$ W/(m²·K) (água interna)
- $h_e = 50$ W/(m²·K) (ar externo)
- $R_{f,i} = 0,0001$ m²·K/W (incrustação interna)
- $R_{f,e} = 0,0002$ m²·K/W (incrustação externa)

In [ ]:
# ============================================================
# EXERCÍCIO 4.1: Coeficiente Global U
# ============================================================

print("=" * 60)
print("EXERCÍCIO 4.1: COEFICIENTE GLOBAL U")
print("=" * 60)

# Dados
k_aco = 45.0         # W/(m·K)
D_i = 0.020          # m
D_e = 0.025          # m
h_i = 3500.0         # W/(m²·K)
h_e = 50.0           # W/(m²·K)
R_fi = 0.0001        # m²·K/W
R_fe = 0.0002        # m²·K/W

# Resistências (referenciadas à área externa)
R_conv_i = 1 / (h_i * (D_i / D_e))  # referenciada à área externa
R_fouling_i = R_fi * (D_e / D_i)
R_wall = (D_e * np.log(D_e / D_i)) / (2 * k_aco)
R_fouling_e = R_fe
R_conv_e = 1 / h_e

R_total = R_conv_i + R_fouling_i + R_wall + R_fouling_e + R_conv_e
U = 1 / R_total

print(f"\n📊 Dados do problema:")
print(f"  • Tubo: D_i = {D_i*1000:.0f} mm, D_e = {D_e*1000:.0f} mm, k = {k_aco} W/(m·K)")
print(f"  • h_i = {h_i:.0f} W/(m²·K), h_e = {h_e:.0f} W/(m²·K)")
print(f"  • R_f,i = {R_fi:.4f} m²·K/W, R_f,e = {R_fe:.4f} m²·K/W")

print(f"\n🧮 Resistências térmicas (ref. área externa):")
print(f"  • R_conv,i = {R_conv_i:.5f} K·m²/W ({R_conv_i/R_total*100:.1f}%)")
print(f"  • R_fouling,i = {R_fouling_i:.5f} K·m²/W ({R_fouling_i/R_total*100:.1f}%)")
print(f"  • R_wall = {R_wall:.5f} K·m²/W ({R_wall/R_total*100:.1f}%)")
print(f"  • R_fouling,e = {R_fouling_e:.5f} K·m²/W ({R_fouling_e/R_total*100:.1f}%)")
print(f"  • R_conv,e = {R_conv_e:.5f} K·m²/W ({R_conv_e/R_total*100:.1f}%)")
print(f"  • R_total = {R_total:.5f} K·m²/W")

print(f"\n🔍 Coeficiente global:")
print(f"  • U = 1/R_total = {U:.1f} W/(m²·K)")

print(f"\n💡 Interpretação:")
print(f"  A convecção externa (ar) domina a resistência ({R_conv_e/R_total*100:.1f}%),")
print(f"  pois h_e é muito menor que h_i. U ≈ h_e neste caso.")

---

## 🌊 Seção 5: Correlações Convectivas

### Exercício 5.1: Dittus-Boelter

Estime $h$ para água ($\mathrm{Pr} = 4,3$) escoando a $U = 1,2$ m/s em tubo de $D = 20$ mm, $\mathrm{Re} = 25.000$. Use Dittus-Boelter para resfriamento.

In [ ]:
# ============================================================
# EXERCÍCIO 5.1: Dittus-Boelter
# ============================================================

print("=" * 60)
print("EXERCÍCIO 5.1: CORRELAÇÃO DE DITTUS-BOELTER")
print("=" * 60)

# Dados
Re = 25000.0
Pr = 4.3
D_ex5 = 0.020        # m
k_agua = 0.6         # W/(m·K)
n = 0.3              # resfriamento

# Número de Nusselt
Nu = 0.023 * (Re**0.8) * (Pr**n)

# Coeficiente convectivo
h = Nu * k_agua / D_ex5

print(f"\n📊 Dados do problema:")
print(f"  • Re = {Re:.0f}")
print(f"  • Pr = {Pr}")
print(f"  • D = {D_ex5*1000:.0f} mm")
print(f"  • k_água = {k_agua} W/(m·K)")
print(f"  • n = {n} (resfriamento)")

print(f"\n🧮 Cálculo:")
print(f"  • Nu = 0,023 · Re^0,8 · Pr^n")
print(f"  • Nu = 0,023 × {Re:.0f}^0,8 × {Pr}^{n}")
print(f"  • Nu = {Nu:.1f}")
print(f"\n  • h = Nu · k / D = {Nu:.1f} × {k_agua} / {D_ex5}")
print(f"  • h = {h:.0f} W/(m²·K)")

print(f"\n💡 Interpretação:")
print(f"  O valor h = {h:.0f} W/(m²·K) é típico para água em regime turbulento.")
print(f"  Para comparação:")
print(f"    • Água turbulenta: h ~ 1000-5000 W/(m²·K)")
print(f"    • Óleo turbulento: h ~ 100-500 W/(m²·K)")
print(f"    • Ar turbulento: h ~ 50-200 W/(m²·K)")

---

## 📈 Seção 6: Visualização de Efetividade

Vamos visualizar como a efetividade varia com NTU para diferentes configurações.

In [ ]:
# ============================================================
# VISUALIZAÇÃO: EFETIVIDADE vs. NTU
# ============================================================

NTU_range = np.linspace(0.1, 5, 100)
C_r_values = [0, 0.25, 0.5, 0.75, 1.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Contracorrente
for C_r in C_r_values:
    if C_r == 1:
        epsilon_range = NTU_range / (1 + NTU_range)
    else:
        exp_term = np.exp(-NTU_range * (1 - C_r))
        epsilon_range = (1 - exp_term) / (1 - C_r * exp_term)
    axes[0].plot(NTU_range, epsilon_range, linewidth=2, label=f'C_r = {C_r}')

axes[0].set_xlabel('NTU', fontsize=12)
axes[0].set_ylabel('Efetividade ε', fontsize=12)
axes[0].set_title('Contracorrente', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

# Gráfico 2: Paralelo
for C_r in C_r_values:
    epsilon_range = (1 - np.exp(-NTU_range * (1 + C_r))) / (1 + C_r)
    axes[1].plot(NTU_range, epsilon_range, linewidth=2, label=f'C_r = {C_r}')

axes[1].set_xlabel('NTU', fontsize=12)
axes[1].set_ylabel('Efetividade ε', fontsize=12)
axes[1].set_title('Paralelo', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

print("\n💡 Observações:")
print("  • Contracorrente sempre tem ε maior que paralelo")
print("  • Para NTU → ∞, contracorrente: ε → 1/(1+C_r)")
print("  • Para NTU → ∞, paralelo: ε → 1/(1+C_r)")
print("  • Para C_r = 0 (condensador): ε = 1 - exp(-NTU)")

---

## 🔬 Estudos de Caso

Os estudos de caso aplicam os conceitos deste capítulo em problemas reais de engenharia.

| Estudo de Caso | Descrição | Link |
|----------------|-----------|------|
| **Caso 9.1** | Resfriamento de Óleo Hidráulico em Hidrelétrica | [🔗 Abrir](../casos/09_1_Resfriamento_Oleo_Hidraulico.ipynb) |
| **Caso 9.2** | Condensador de Vapor para UTE de 100 MW | [🔗 Abrir](../casos/09_2_Condensador_Vapor_UTE.ipynb) |
| **Caso 9.3** | Caldeira Industrial para Geração de Vapor | [🔗 Abrir](../casos/09_3_Caldeira_Industrial.ipynb) |

> 💡 **Dica:** Os estudos de caso podem ser executados independentemente deste notebook principal.

---

## 📖 Referências

- INCROPERA, F. P. et al. *Fundamentals of Heat and Mass Transfer*. 8ª ed. Wiley, 2021.
- KAKAÇ, S.; LIU, H.; MAJUMDAR, A. *Heat Exchangers: Selection, Rating, and Thermal Design*. 3ª ed. CRC Press, 2012.
- SHAH, R. K.; SEKULIĆ, D. P. *Fundamentals of Heat Exchanger Design*. Wiley, 2003.
- HEWITT, G. F.; SHIRES, G. L.; BOTT, T. R. *Process Heat Transfer*. CRC Press, 1994.

---

## 🔄 Navegação

- [📚 Capítulo Anterior: Calor em Solos](08_Calor_Solos.ipynb)
- [📚 Próximo Capítulo: Aletas e Superfícies Estendidas](10_Aletas_Superficies.ipynb)
- [📂 Outros Capítulos](../notebooks/)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**QR Code do Capítulo 9**  
Aponte seu celular para o QR Code no livro para acessar este notebook!

</div>

In [ ]:
print("=" * 60)
print("✅ NOTEBOOK CONCLUÍDO COM SUCESSO!")
print("=" * 60)
print("\n🎓 Você completou o Capítulo 9!")
print("📖 Próximo passo: Capítulo 10 - Aletas e Superfícies Estendidas")
print("\n💡 Dica: Execute este notebook quantas vezes precisar!")
print("   Modifique os parâmetros e explore diferentes cenários.")